In [1]:
!pip install qiskit qiskit-aer --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00


In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
import random

In [3]:
n = 20

alice_bits = [random.randint(0,1) for _ in range(n)]
alice_bases = [random.choice(['Z','X']) for _ in range(n)]

eve_bases = [random.choice(['Z','X']) for _ in range(n)]
bob_bases = [random.choice(['Z','X']) for _ in range(n)]

bob_results = []

sim = Aer.get_backend('aer_simulator')

for i in range(n):
    qc = QuantumCircuit(1,1)


    if alice_bits[i] == 1:
        qc.x(0)

    if alice_bases[i] == 'X':
        qc.h(0)


    if eve_bases[i] == 'X':
        qc.h(0)

    qc.measure(0,0)

    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1).result()
    eve_result = int(list(result.get_counts().keys())[0])


    qc = QuantumCircuit(1,1)

    if eve_result == 1:
        qc.x(0)

    if eve_bases[i] == 'X':
        qc.h(0)

    if bob_bases[i] == 'X':
        qc.h(0)

    qc.measure(0,0)

    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1).result()
    bob_bit = int(list(result.get_counts().keys())[0])

    bob_results.append(bob_bit)

In [4]:
key_indices = [i for i in range(n) if alice_bases[i] == bob_bases[i]]

alice_key = [alice_bits[i] for i in key_indices]
bob_key = [bob_results[i] for i in key_indices]
errors = sum(1 for a,b in zip(alice_key, bob_key) if a != b)
error_rate = errors / len(alice_key) if len(alice_key) > 0 else 0

print("Alice Bits:  ", alice_bits)
print("Alice Bases: ", alice_bases)
print("Eve Bases:   ", eve_bases)
print("Bob Bases:   ", bob_bases)

print("\nFinal Key (Alice):", alice_key)
print("Final Key (Bob):  ", bob_key)

print("\nError Rate:", error_rate)

if error_rate > 0.2:
    print("Eve is there! Communication NOT secure.")
else:
    print("Secure communication.")

Alice Bits:   [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1]
Alice Bases:  ['X', 'X', 'Z', 'X', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'X', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'X', 'X', 'X']
Eve Bases:    ['X', 'Z', 'X', 'X', 'Z', 'Z', 'X', 'X', 'Z', 'Z', 'X', 'Z', 'Z', 'X', 'Z', 'Z', 'Z', 'Z', 'X', 'X']
Bob Bases:    ['Z', 'X', 'X', 'Z', 'X', 'X', 'Z', 'Z', 'Z', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'Z', 'Z', 'X', 'Z', 'X']

Final Key (Alice): [1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1]
Final Key (Bob):   [1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1]

Error Rate: 0.4166666666666667
Eve is there! Communication NOT secure.
